# 04 — Aggregations and Window Functions

`groupBy`/`agg` patterns, window functions (rank, lag/lead, running totals), pivoting, and the classic "top-N per group" interview problem.

> **Setup note:** these notebooks are written but **not executed** — PySpark is not
> installed in this environment. To run them locally:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate
> pip install pyspark==3.5.1
> # Java 11/17 must be on PATH (java -version)
> jupyter notebook
> ```
>
> Everything below is correct, runnable PySpark — read it as a reference and run
> cell-by-cell once your environment is set up.

In [ ]:
sales = spark.createDataFrame(
    [
        ("east", "alice", "2024-01", 100),
        ("east", "alice", "2024-02", 150),
        ("east", "bob",   "2024-01", 200),
        ("east", "bob",   "2024-02", 90),
        ("west", "cara",  "2024-01", 300),
        ("west", "cara",  "2024-02", 250),
        ("west", "dan",   "2024-01", 120),
    ],
    ["region", "rep", "month", "amount"],
)

## 1. `groupBy` + `agg` patterns

`groupBy` alone is lazy and returns a `GroupedData` object — it only becomes a DataFrame once you call `.agg(...)` (or a shorthand like `.count()`). Use `.agg()` with multiple aggregate expressions to compute several metrics in a single shuffle instead of one `groupBy` per metric.

In [ ]:
from pyspark.sql.functions import sum as spark_sum, avg, max as spark_max, count, round as spark_round

region_summary = sales.groupBy("region").agg(
    spark_sum("amount").alias("total_amount"),
    spark_round(avg("amount"), 2).alias("avg_amount"),
    spark_max("amount").alias("max_amount"),
    count("*").alias("n_rows"),
)
region_summary.orderBy("region").show()

## 2. Window functions

A window function computes a value **per row** using a frame of related rows, without collapsing rows the way `groupBy` does (rank, running total, moving average, previous/next row). Define a `Window` spec with `partitionBy` (like a `GROUP BY`, but rows aren't collapsed) and `orderBy` (defines row order within each partition, needed for ranking/lag/lead/cumulative functions).

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import rank, dense_rank, row_number, lag, lead, sum as spark_sum2

by_region_amount_desc = Window.partitionBy("region").orderBy(col("amount").desc())

ranked = sales.withColumn("rank", rank().over(by_region_amount_desc)) \
              .withColumn("dense_rank", dense_rank().over(by_region_amount_desc)) \
              .withColumn("row_number", row_number().over(by_region_amount_desc))
ranked.orderBy("region", "rank").show()

**`rank` vs `dense_rank` vs `row_number` — a favorite interview distinction:**

- `row_number()` — always unique, sequential (1,2,3,4...), ties broken arbitrarily (by whatever order the ties happen to land in).
- `rank()` — ties get the *same* rank, but the next rank **skips** (1,1,3,4...) — like Olympic medal ranking with a gap after ties.
- `dense_rank()` — ties get the same rank, next rank does **not** skip (1,1,2,3...).

In [ ]:
by_rep_month = Window.partitionBy("rep").orderBy("month")

running = sales.withColumn(
    "running_total", spark_sum2("amount").over(by_rep_month.rowsBetween(Window.unboundedPreceding, Window.currentRow))
).withColumn(
    "prev_month_amount", lag("amount", 1).over(by_rep_month)
).withColumn(
    "next_month_amount", lead("amount", 1).over(by_rep_month)
)
running.orderBy("rep", "month").show()

**Frame specs:** `rowsBetween(start, end)` counts physical rows (e.g. "last 3 rows" for a moving average); `rangeBetween(start, end)` counts by the *value* in the `orderBy` column (e.g. "all rows within 7 of the current value") — useful for time-based windows on numeric/date columns. `Window.unboundedPreceding` / `unboundedFollowing` mean "from the start/to the end of the partition".

## 3. Pivoting

`groupBy(...).pivot(col).agg(...)` turns distinct values of a column into new columns — the Spark equivalent of a spreadsheet pivot table. Passing the list of distinct values explicitly (`pivot("month", ["2024-01", "2024-02"])`) avoids Spark having to scan the data once just to discover them.

In [ ]:
pivoted = sales.groupBy("rep").pivot("month", ["2024-01", "2024-02"]).agg(spark_sum("amount"))
pivoted.show()

## 4. Classic interview problem: top-N per group

"For each region, find the rep with the highest total sales." This is the single most common Spark/SQL interview question — solved with a window function, **not** a `groupBy` + `limit` (which can't be scoped per group).

In [ ]:
rep_totals = sales.groupBy("region", "rep").agg(spark_sum("amount").alias("total"))

w = Window.partitionBy("region").orderBy(col("total").desc())
top_rep_per_region = (
    rep_totals
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)
top_rep_per_region.show()
# generalizes directly to "top 3 per group": .filter(col("rn") <= 3)

## 5. Interview Q&A

1. **"How do you get the top 3 rows per group?"** — window function with `row_number()` (or `rank()`/`dense_rank()` if ties should all be included), `partitionBy` the group key, `orderBy` the ranking metric descending, then `filter(rn <= 3)`.
2. **"Why not just `groupBy(group_col).agg(...).orderBy(...).limit(n)`?"** — that limits the *entire result*, not n rows *per group*.
3. **"Are window functions a shuffle?"** — yes, `partitionBy` requires repartitioning data by the partition key (similar cost to a `groupBy`), so avoid unnecessary/wide window operations on very large datasets.
4. **"What does `pivot` cost that a plain `groupBy` doesn't?"** — if you don't pass the distinct pivot values explicitly, Spark runs an extra job just to discover them before the real aggregation.

## Summary

- `agg()` with multiple expressions computes several metrics in one shuffle.
- Window functions add per-row computed columns without collapsing rows; `row_number`/`rank`/`dense_rank` differ in how they handle ties.
- Top-N-per-group = window function + `row_number` + filter, not `groupBy`+`limit`.
- Next: `05_partitioning_and_performance_tuning.ipynb`.